In [ ]:
# Install Packages Here
import pandas as pd
import torch
%pip install pyreadr
import pyreadr
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

In [ ]:
# Data Prep Methods
def readRSD(url, scratch):
  '''
  Reads in an RSD file from url and returns a dataframe.
  Scratch should be in ./_______.rda format.
  '''
  local = pyreadr.download_file(url, scratch)
  result = pyreadr.read_r(local)
  df = result[None]
  return df

def get_team_stats(df, team_id, year):
    """
    Returns all stats for a given team from a Detailed Results DataFrame,
    normalized to team/opponent perspective (instead of winner/loser).

    Parameters:
        df      : Detailed Results DataFrame
        team_id : The team's ID to filter by
        year    : (optional) Season year to filter by. If None, returns all years.

    Returns:
        DataFrame with one row per game, stats from the team's perspective
    """

    stat_cols = ['FGM', 'FGA', 'FGM3', 'FGA3', 'FTM', 'FTA',
                 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']


    won = df[(df['WTeamID'] == team_id) & (df['Season'] == year)].copy()
    lost = df[(df['LTeamID'] == team_id) & (df['Season'] == year)].copy()

    # Games where team won
    won['TeamID']      = won['WTeamID']
    won['OpponentID']  = won['LTeamID']
    won['TeamScore']   = won['WScore']
    won['OppScore']    = won['LScore']
    won['Result']      = 1
    for col in stat_cols:
        won[col]           = won[f'W{col}']
        won[f'Opp{col}']   = won[f'L{col}']

    # Games where team lost
    lost['TeamID']     = lost['LTeamID']
    lost['OpponentID'] = lost['WTeamID']
    lost['TeamScore']  = lost['LScore']
    lost['OppScore']   = lost['WScore']
    lost['Result']     = 0
    for col in stat_cols:
        lost[col]          = lost[f'L{col}']
        lost[f'Opp{col}']  = lost[f'W{col}']

    # --- Combine and select final columns ---
    keep_cols = ['Season', 'DayNum', 'TeamID', 'OpponentID', 'TeamScore',
                 'OppScore', 'Result', 'NumOT'] + \
                stat_cols + [f'Opp{col}' for col in stat_cols]

    team_games = pd.concat([won, lost])[keep_cols]

    team_games = team_games.astype(float)

    team_games = team_games.sort_values(['Season', 'DayNum']).reset_index(drop=True)

    team_games = pd.concat([team_games, team_games.mean(axis=0).to_frame().T], ignore_index=True)


    return team_games

def get_tensor(df):
  '''
  Takes boxscore df  for a team over a season as input and returns a
  single tensor containing their season aggregated stats.
  '''
  df = df.drop(columns=['Season', 'DayNum', 'TeamID', 'OpponentID'])
  # convert to a NumPy array first
  stat_tensor = torch.tensor(df.iloc[-1].to_numpy(), dtype=torch.float32)
  return stat_tensor



# Get all results from 2003 onwards
def get_all_results(outcomes):
  '''
  Scrapes all game outcomes from df and returns a boolean vector of game outcomes.
  This method gives us our prediction targets. A value of 1 corresponds to the
  team with lower team ID winning and 0 to the team with higher team ID.
  Note that seeds are not a factor in determining 0 or 1, so this has nothing
  to do with seeding.
  '''
  outcomes = outcomes[outcomes['Season'] >= 2003] # Only have box score stats from 2003 onwards
  results = []
  for i in range(len(outcomes)):

    if outcomes.iloc[i]['WTeamID'] < outcomes.iloc[i]['LTeamID']:
      results.append(1)
    else:
      results.append(0)

  results = torch.tensor(results)
  return results


def gather_team_stats(outcomes,box_scores):
  '''
  Takes outcomes, a df containing each MM game's winner and loser, alongside
  box_scores, a df containing the box scores for all games from 2003 onwards,
  and combines them into a tensor where each row contains all season stats
  for two teams in the tournament. This method returns the X tensor we need
  for ML methods.
  '''
  outcomes = outcomes[outcomes['Season'] >= 2003]
  feature_list = []
  for i in range(len(outcomes)):

    team1 = min(outcomes['WTeamID'].iloc[i], outcomes['LTeamID'].iloc[i])
    team2 = max(outcomes['WTeamID'].iloc[i], outcomes['LTeamID'].iloc[i])

    df1 = get_team_stats(box_scores, team1, outcomes['Season'].iloc[i])
    df2 = get_team_stats(box_scores, team2, outcomes['Season'].iloc[i])

    tens1 = get_tensor(df1)
    tens2 = get_tensor(df2)
    # revise to use feature differences, not raw team stats
    tens = tens1 - tens2
    feature_list.append(tens)

  return torch.stack(feature_list)

# Read Files
outcomes = readRSD('https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/DataFiles/NCAATourneyCompactResults.rds', './TourneyResults.rda')
box_scores = readRSD('https://github.com/kim3-sudo/march_madness_data/raw/refs/heads/main/Prelim2019_RegularSeasonDetailedResults.rds', './DetailedStats.rda')

# Assemble Data
X = gather_team_stats(outcomes, box_scores)
y = get_all_results(outcomes)

In [ ]:
from torch import nn
import torch

class LogisticRegression(nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.layer1 = nn.Linear(n_features, 2)

    def forward(self, X):
        return self.layer1(X)

In [ ]:
def train_model(model, X, y, n_epochs=5000):  
    opt = torch.optim.SGD(model.parameters(), lr=0.01)
    loss_fn = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(n_epochs): 
        opt.zero_grad()
        y_pred = model(X)
        loss = loss_fn(y_pred, y.long())
        losses.append(loss.item())
        loss.backward()
        opt.step()
    return losses

# Split data the same way as FirstPassClassification.ipynb
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# revised to standardize features
scaler = StandardScaler()

# convert to numpy for sklearn
X_train_np = X_train.numpy()
X_test_np  = X_test.numpy()

# fit on train, transform both
X_train = torch.tensor(scaler.fit_transform(X_train_np)).float()
X_test  = torch.tensor(scaler.transform(X_test_np)).float()

# Train on the training set
model = LogisticRegression(n_features=X_train.shape[1])
losses = train_model(model, X_train.float(), y_train.float())

# Evaluate on the test set
with torch.no_grad():
    logits = model(X_test.float())
    preds = torch.argmax(logits, dim=1)

print(accuracy_score(y_test, preds))

In [ ]:
import matplotlib.pyplot as plt

plt.plot(losses)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.show()